# minilake + real Spark + real Delta Lake

This notebook writes a real Delta Lake table with real Spark, then reads the *same*
files back through minilake's SQL Statement Execution API — no copy, no sync step.

The Spark code does not run in this kernel. It is submitted to minilake's Jobs API,
which executes it with `spark-submit` in a sibling Spark container — the same path a
real job takes. That is deliberate: on Databricks your notebook runs on a cluster,
not inside the control plane, and it keeps this image free of a second Spark.

Everything below talks to the minilake that launched this notebook.


## 0. Connect

`MINILAKE_HOST` and `MINILAKE_DATA_DIR` are injected by minilake when it starts JupyterLab.


In [ ]:
import os

HOST = os.environ.get("MINILAKE_HOST", "http://127.0.0.1:8000")
DATA_DIR = os.environ.get("MINILAKE_DATA_DIR", "/data")

from databricks.sdk import WorkspaceClient

w = WorkspaceClient(host=HOST, token="dev")
print("connected to", HOST, "| user:", w.current_user.me().user_name)


## 1. A helper that runs PySpark for real

Stages the script in the workspace, creates a one-off job with a `spark_python_task`,
runs it, waits, and hands back stdout. This is exactly what the MCP tool
`run_python_script` does — see `minilake://pyspark-guide` for the full recipe.

`delta=True` adds the Delta jars: the base Spark image ships none, so without it
`format("delta")` fails with `DATA_SOURCE_NOT_FOUND`.


In [ ]:
import base64, time, uuid, requests

DELTA_PACKAGE = "io.delta:delta-spark_2.12:3.2.1"


def run_python_script(script: str, name: str = "nb", delta: bool = True, timeout: int = 600):
    """Run PySpark in a sibling Spark container and return its captured output."""
    tag = f"{name}-{uuid.uuid4().hex[:8]}"
    path = f"/Shared/notebook/{tag}.py"

    requests.post(
        f"{HOST}/api/2.0/workspace/import",
        json={
            "path": path,
            "content": base64.b64encode(script.encode()).decode(),
            "language": "PYTHON",
            "format": "SOURCE",
            "overwrite": True,
        },
    ).raise_for_status()

    task = {"task_key": "main", "spark_python_task": {"python_file": path}}
    if delta:
        task["libraries"] = [{"maven": {"coordinates": DELTA_PACKAGE}}]

    job_id = requests.post(
        f"{HOST}/api/2.2/jobs/create", json={"name": tag, "tasks": [task]}
    ).json()["job_id"]
    run_id = requests.post(
        f"{HOST}/api/2.2/jobs/run-now", json={"job_id": job_id}
    ).json()["run_id"]

    deadline = time.time() + timeout
    while time.time() < deadline:
        run = requests.get(f"{HOST}/api/2.2/jobs/runs/get", params={"run_id": run_id}).json()
        if (run.get("state") or {}).get("life_cycle_state") == "TERMINATED":
            break
        time.sleep(3)
    else:
        raise TimeoutError(f"run {run_id} did not finish in {timeout}s")

    out = requests.get(f"{HOST}/api/2.2/jobs/runs/get-output", params={"run_id": run_id}).json()
    result = (run.get("state") or {}).get("result_state")
    print(f"run {run_id}: {result}")
    if out.get("error"):
        print(out["error"])
    print(out.get("logs") or "")
    return result


## 2. Create a catalog and schema

Reset first, so the notebook can be re-run from scratch at any time.


In [ ]:
requests.post(f"{HOST}/_minilake/reset")

w.catalogs.create(name="notebook_demo")
w.schemas.create(name="events", catalog_name="notebook_demo")
print([c.name for c in w.catalogs.list()])


## 3. Write a real Delta table with real Spark

`storage_location` is on the data volume shared with the job container, so the files
Spark writes are the very files minilake will read back.


In [ ]:
storage_location = f"{DATA_DIR}/delta/notebook_demo/events/people"

script = f'''
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("minilake-notebook")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

df = spark.createDataFrame(
    [(1, "Ada", 36), (2, "Grace", 45), (3, "Alan", 41)],
    "id INT, name STRING, age INT",
)
df.write.format("delta").mode("overwrite").save("{storage_location}")
print("wrote", df.count(), "rows to", "{storage_location}")
'''

run_python_script(script, name="write-delta")


## 4. Register it as an EXTERNAL Delta table

minilake stores only the *metadata* — the files stay exactly where Spark put them.
EXTERNAL + DELTA is what makes minilake read through `delta_scan()` instead of
looking for a DuckDB table.


In [ ]:
from databricks.sdk.service.catalog import DataSourceFormat, TableType

w.tables.create(
    name="people",
    catalog_name="notebook_demo",
    schema_name="events",
    table_type=TableType.EXTERNAL,
    data_source_format=DataSourceFormat.DELTA,
    storage_location=storage_location,
)
print(w.tables.get("notebook_demo.events.people").table_type)


## 5. Query the *same* data through minilake's SQL API

No copy, no sync step — DuckDB's `delta` extension reads the real Delta files Spark
just wrote.


In [ ]:
wh = w.warehouses.create(name="notebook_wh")

result = w.statement_execution.execute_statement(
    warehouse_id=wh.id,
    statement="SELECT * FROM notebook_demo.events.people ORDER BY id",
)
print(result.status.state)
for row in result.result.data_array:
    print(row)


## 6. Round-trip: read it back with Spark too

Same files, three access paths — Spark, minilake SQL, and SDK-managed metadata.


In [ ]:
run_python_script(f'''
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("minilake-readback")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.read.format("delta").load("{storage_location}").orderBy("id").show()
''', name="read-delta")


---

### Where to go next

- **Query editor** (`/ui/`) — the same SQL, with completion fed by this catalog.
- **Jobs** (`/ui/jobs`) — every run above is listed there, with its logs.
- **Data catalog** (`/ui/catalog`) — `notebook_demo.events.people`, its columns and DDL.

Only **EXTERNAL Delta** tables are visible to Spark. A MANAGED table is a DuckDB table
with no files, so query those through the SQL API instead.
